# Notebook 0.5 – Enriquecimento de Obras (Produtoras & Distribuidoras)

**Objetivo**: Construir `df_obras_enriquecido.parquet` contendo features estruturadas de produtoras e distribuidoras para cada obra (CPB/ROE).

**Output**: Base auxiliar com 9 features numéricas + listas normalizadas para uso nos Notebooks 1 e 2.

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
import json
import unicodedata
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print(f"Polars version: {pl.__version__}")
print(f"Pandas version: {pd.__version__}")

Polars version: 1.35.1
Pandas version: 2.3.3


## 1. Funções Auxiliares

In [2]:
def normalizar_texto(texto):
    """Normaliza texto: lowercase, remove acentos, espaços extras"""
    if pd.isna(texto) or texto == '':
        return None
    
    # Converter para string
    texto = str(texto)
    
    # Lowercase
    texto = texto.lower()
    
    # Remover acentos
    texto = unicodedata.normalize('NFKD', texto)
    texto = texto.encode('ASCII', 'ignore').decode('ASCII')
    
    # Remover espaços extras e caracteres especiais
    texto = re.sub(r'\s+', ' ', texto)
    texto = texto.strip()
    
    return texto if texto else None


def identificar_major(nome_dist):
    """Identifica se distribuidora é major (Disney, Warner, Universal, Sony, Paramount)"""
    if pd.isna(nome_dist):
        return False
    
    nome_norm = normalizar_texto(nome_dist)
    if nome_norm is None:
        return False
    
    majors = ['disney', 'warner', 'universal', 'sony', 'paramount']
    
    return any(major in nome_norm for major in majors)


def calcular_percentil_grupo(valor, percentis):
    """Classifica valor em grupos de percentis (1-5)"""
    if pd.isna(valor):
        return None
    
    if valor <= percentis[0.2]:
        return 1
    elif valor <= percentis[0.4]:
        return 2
    elif valor <= percentis[0.6]:
        return 3
    elif valor <= percentis[0.8]:
        return 4
    else:
        return 5


print("✓ Funções auxiliares definidas")

✓ Funções auxiliares definidas


## 2. Importação dos Conjuntos ANCINE

In [3]:
# URLs dos datasets ANCINE
urls = {
    'agentes': 'https://dados.ancine.gov.br/dados-abertos/agentes-economicos-regulares.csv',
    'produtoras_indep': 'https://dados.ancine.gov.br/dados-abertos/produtoras-independentes.csv',
    'produtores_cpb': 'https://dados.ancine.gov.br/dados-abertos/produtores-de-obras-nao-publicitarias-brasileiras.csv',
    'produtores_roe': 'https://dados.ancine.gov.br/dados-abertos/produtores-de-obras-nao-publicitarias-estrangeiras.csv',
    'lancamentos': 'https://dados.ancine.gov.br/dados-abertos/lancamentos-comerciais-por-distribuidoras.csv'
}

print("Iniciando importação dos conjuntos ANCINE...\n")

Iniciando importação dos conjuntos ANCINE...



In [4]:
# 2.0. Carregar lista de obras da base de sessões

print("Carregando lista de obras da base de sessões...\n")

# Caminho relativo: ../bases/df_sessoes_limpo.parquet
sessoes_path = Path('..') / 'bases' / 'df_sessoes_limpo.parquet'

df_sessoes = pl.read_parquet(sessoes_path)

# Extrair lista única de CPB_ROE
obras_validas = df_sessoes.select('CPB_ROE').unique().to_pandas()['CPB_ROE'].tolist()

print(f"✓ Obras únicas na base de sessões: {len(obras_validas):,}")
print(f"  Exemplo: {obras_validas[:3]}\n")

Carregando lista de obras da base de sessões...

✓ Obras únicas na base de sessões: 2,197
  Exemplo: ['E2500212900000', 'E2400062000000', 'B0500384200000']



### 2.1. Conjunto 1: Agentes Econômicos Regulares

In [5]:
print("[1/5] Carregando Agentes Econômicos Regulares...")

df_agentes = pl.read_csv(
    urls['agentes'],
    separator=';',
    encoding='utf-8',
    infer_schema_length=10000,
    truncate_ragged_lines=True,
    ignore_errors=True
)

# Padronizar nomes de colunas
df_agentes = df_agentes.rename({
    col: col.lower().replace(' ', '_').replace('º', 'n') 
    for col in df_agentes.columns
})

print(f"  Shape: {df_agentes.shape}")
print(f"  Colunas: {df_agentes.columns}")
print(f"  ✓ Agentes carregados\n")

[1/5] Carregando Agentes Econômicos Regulares...
  Shape: (18822, 11)
  Colunas: ['registro_ancine', 'data_registro', 'razao_social', 'cnpj', 'data_constituicao', 'uf', 'municipio', 'codigo_municipio_ibge', 'classificacao_agente_economico', 'natureza_juridica', 'brasileiro_independente']
  ✓ Agentes carregados



### 2.2. Conjunto 3: Produtoras Independentes Regulares (contém Nível 1-5)

In [6]:
print("[2/5] Carregando Produtoras Independentes Regulares (Nível 1-5)...")

df_produtoras_indep = pl.read_csv(
    urls['produtoras_indep'],
    separator=';',
    encoding='utf-8',
    infer_schema_length=10000,
    truncate_ragged_lines=True,
    ignore_errors=True
)

# Padronizar nomes de colunas
df_produtoras_indep = df_produtoras_indep.rename({
    col: col.lower().replace(' ', '_').replace('º', 'n') 
    for col in df_produtoras_indep.columns
})

print(f"  Shape: {df_produtoras_indep.shape}")
print(f"  Colunas: {df_produtoras_indep.columns}")
print(f"  ✓ Produtoras independentes carregadas\n")

[2/5] Carregando Produtoras Independentes Regulares (Nível 1-5)...
  Shape: (12000, 6)
  Colunas: ['razao_social', 'registro_ancine', 'cnpj', 'municipio', 'uf', 'classificacao_nivel_produtora']
  ✓ Produtoras independentes carregadas



### 2.3. Conjunto 7: Produtores de Obras Brasileiras (CPB)

In [7]:
print("[3/5] Carregando Produtores de Obras Brasileiras (CPB)...")

df_produtores_cpb = pl.read_csv(
    urls['produtores_cpb'],
    separator=';',  # <- IMPORTANTE: CSV usa ponto-e-vírgula
    encoding='utf-8',
    infer_schema_length=10000,
    truncate_ragged_lines=True,
    ignore_errors=True
)

# Padronizar nomes de colunas
df_produtores_cpb = df_produtores_cpb.rename({
    col: col.lower().replace(' ', '_').replace('º', 'n') 
    for col in df_produtores_cpb.columns
})

print(f"  Shape: {df_produtores_cpb.shape}")
print(f"  Colunas: {df_produtores_cpb.columns}")
print(f"  ✓ Produtores CPB carregados\n")

[3/5] Carregando Produtores de Obras Brasileiras (CPB)...
  Shape: (71211, 5)
  Colunas: ['produtor', 'cnpj_produtor', 'pais_produtor', 'titulo_original', 'cpb']
  ✓ Produtores CPB carregados



### 2.4. Conjunto 15: Produtores de Obras Estrangeiras (ROE)

In [8]:
print("[4/5] Carregando Produtores de Obras Estrangeiras (ROE)...")

df_produtores_roe = pl.read_csv(
    urls['produtores_roe'],
    separator=';',
    encoding='utf-8',
    infer_schema_length=10000,
    truncate_ragged_lines=True,
    ignore_errors=True
)

# Padronizar nomes de colunas
df_produtores_roe = df_produtores_roe.rename({
    col: col.lower().replace(' ', '_').replace('º', 'n') 
    for col in df_produtores_roe.columns
})

print(f"  Shape: {df_produtores_roe.shape}")
print(f"  Colunas: {df_produtores_roe.columns}")
print(f"  ✓ Produtores ROE carregados\n")

[4/5] Carregando Produtores de Obras Estrangeiras (ROE)...
  Shape: (95456, 3)
  Colunas: ['produtor', 'titulo_original', 'roe']
  ✓ Produtores ROE carregados



### 2.5. Lançamentos Comerciais por Distribuidoras

In [9]:
print("[5/5] Carregando Lançamentos Comerciais por Distribuidoras...")

df_lancamentos = pl.read_csv(
    urls['lancamentos'],
    separator=';',
    encoding='utf-8',
    infer_schema_length=10000,
    truncate_ragged_lines=True,
    ignore_errors=True
)

# Padronizar nomes de colunas
df_lancamentos = df_lancamentos.rename({
    col: col.lower().replace(' ', '_').replace('º', 'n').replace('/', '_') 
    for col in df_lancamentos.columns
})

print(f"  Shape: {df_lancamentos.shape}")
print(f"  Colunas: {df_lancamentos.columns}")
print(f"  ✓ Lançamentos carregados\n")

print("=" * 60)
print("✓ Todos os conjuntos importados com sucesso")
print("=" * 60)

[5/5] Carregando Lançamentos Comerciais por Distribuidoras...
  Shape: (6828, 10)
  Colunas: ['data_lancamento_obra', 'titulo_original', 'cpb_roe', 'tipo_obra', 'pais_obra', 'publico_total', 'renda_total', 'razao_social_distribuidora', 'registro_distribuidora', 'cnpj_distribuidora']
  ✓ Lançamentos carregados

✓ Todos os conjuntos importados com sucesso


## 3. Limpeza e Preparação de Lançamentos (Distribuidoras)

In [10]:
print("\nIniciando limpeza de Lançamentos Comerciais...\n")

# Converter para Pandas para facilitar manipulações
df_lanc = df_lancamentos.to_pandas()

print(f"Shape inicial: {df_lanc.shape}")

# REGRA 1: Excluir obra ROE = E1300000100000
obras_antes = df_lanc['cpb_roe'].nunique()
df_lanc = df_lanc[df_lanc['cpb_roe'] != 'E1300000100000']
obras_depois = df_lanc['cpb_roe'].nunique()
print(f"\n[Regra 1] Obras removidas (ROE E1300000100000): {obras_antes - obras_depois}")

# REGRA 2 e 3: Tratar distribuidoras problemáticas
dist_problematicas = [
    'KOMPANHIA TEATRO MULTIMÃDIA DE SÃƒO PAULO',
    'CINE CAXAMBU LTDA'
]

# Normalizar nomes de distribuidoras
df_lanc['razao_social_dist_norm'] = df_lanc['razao_social_distribuidora'].apply(normalizar_texto)
dist_problematicas_norm = [normalizar_texto(d) for d in dist_problematicas]

print(f"\n[Regra 2 e 3] Tratando distribuidoras problemáticas...")

# Identificar obras com cada distribuidora
for obra in df_lanc['cpb_roe'].unique():
    obra_dists = df_lanc[df_lanc['cpb_roe'] == obra]['razao_social_dist_norm'].tolist()
    
    # Contar distribuidoras da obra
    num_dists = len(obra_dists)
    
    # Se tem distribuidora problemática
    tem_problematica = any(d in dist_problematicas_norm for d in obra_dists)
    
    if tem_problematica:
        if num_dists > 1:
            # Remover apenas a distribuidora problemática (co-distribuição)
            df_lanc = df_lanc[
                ~((df_lanc['cpb_roe'] == obra) & 
                  (df_lanc['razao_social_dist_norm'].isin(dist_problematicas_norm)))
            ]
        # Se for única distribuidora, manter (não fazer nada)

print(f"Shape após limpeza: {df_lanc.shape}")
print(f"Obras únicas: {df_lanc['cpb_roe'].nunique()}")
print(f"Distribuidoras únicas: {df_lanc['razao_social_dist_norm'].nunique()}")
print("\n✓ Limpeza concluída")


Iniciando limpeza de Lançamentos Comerciais...

Shape inicial: (6828, 10)

[Regra 1] Obras removidas (ROE E1300000100000): 1

[Regra 2 e 3] Tratando distribuidoras problemáticas...
Shape após limpeza: (6785, 11)
Obras únicas: 6621
Distribuidoras únicas: 383

✓ Limpeza concluída


## 4. Cálculo de Porte das Distribuidoras (2012 até hoje)

In [11]:
print("\nCalculando porte das distribuidoras (2012+)...\n")

# Extrair ano da data de lançamento
# Tentar diferentes formatos de coluna de data
date_cols = [col for col in df_lanc.columns if 'data' in col.lower()]
print(f"Colunas de data encontradas: {date_cols}")

# Usar primeira coluna de data disponível
if date_cols:
    date_col = date_cols[0]
    df_lanc['ano_lancamento'] = pd.to_datetime(
        df_lanc[date_col], 
        errors='coerce'
    ).dt.year
else:
    # Se não houver coluna de data, tentar usar ano cinematográfico
    ano_cols = [col for col in df_lanc.columns if 'ano' in col.lower()]
    if ano_cols:
        df_lanc['ano_lancamento'] = pd.to_numeric(df_lanc[ano_cols[0]], errors='coerce')
    else:
        print("ATENÇÃO: Não foi possível identificar coluna de data/ano. Usando todos os registros.")
        df_lanc['ano_lancamento'] = 2020  # Valor default

# Filtrar período 2012+
df_lanc_filtrado = df_lanc[df_lanc['ano_lancamento'] >= 2012].copy()
print(f"Registros após filtro 2012+: {len(df_lanc_filtrado)}")

# Contar lançamentos totais por distribuidora (2012+)
lancamentos_por_dist = df_lanc_filtrado.groupby('razao_social_dist_norm').agg({
    'cpb_roe': 'nunique'  # Contar obras únicas
}).reset_index()
lancamentos_por_dist.columns = ['distribuidora_norm', 'total_lancamentos']

print(f"\nDistribuidoras únicas (2012+): {len(lancamentos_por_dist)}")
print(f"\nEstatísticas de lançamentos:")
print(lancamentos_por_dist['total_lancamentos'].describe())

# CLASSIFICAÇÃO DE PORTE (limites absolutos)
print(f"\n{'='*60}")
print("CLASSIFICAÇÃO DE PORTE (limites absolutos):")
print("="*60)

def classificar_porte_absoluto(total_lancamentos):
    """
    Classifica porte por limites absolutos mais realistas:
    - Porte 1: até 5 lançamentos (micro)
    - Porte 2: 6-15 lançamentos (pequena)
    - Porte 3: 16-40 lançamentos (média)
    - Porte 4: 41-80 lançamentos (grande)
    - Porte 5: 81+ lançamentos (major)
    """
    if pd.isna(total_lancamentos):
        return None
    if total_lancamentos <= 5:
        return 1
    elif total_lancamentos <= 15:
        return 2
    elif total_lancamentos <= 40:
        return 3
    elif total_lancamentos <= 80:
        return 4
    else:
        return 5

# Aplicar classificação
lancamentos_por_dist['porte'] = lancamentos_por_dist['total_lancamentos'].apply(
    classificar_porte_absoluto
)

print("\nLimites de porte definidos:")
print("  Porte 1 (micro):  até 5 lançamentos")
print("  Porte 2 (pequena): 6-15 lançamentos")
print("  Porte 3 (média):   16-40 lançamentos")
print("  Porte 4 (grande):  41-80 lançamentos")
print("  Porte 5 (major):   81+ lançamentos")

print(f"\nDistribuição de portes (limites absolutos):")
dist_portes = lancamentos_por_dist['porte'].value_counts().sort_index()
for porte, count in dist_portes.items():
    pct = count / len(lancamentos_por_dist) * 100
    print(f"  Porte {porte}: {count:4d} distribuidoras ({pct:5.1f}%)")

print(f"\n✓ Porte das distribuidoras calculado com limites absolutos")


Calculando porte das distribuidoras (2012+)...

Colunas de data encontradas: ['data_lancamento_obra']
Registros após filtro 2012+: 5815

Distribuidoras únicas (2012+): 362

Estatísticas de lançamentos:
count    362.000000
mean      16.063536
std       50.922665
min        1.000000
25%        1.000000
50%        1.000000
75%        4.000000
max      457.000000
Name: total_lancamentos, dtype: float64

CLASSIFICAÇÃO DE PORTE (limites absolutos):

Limites de porte definidos:
  Porte 1 (micro):  até 5 lançamentos
  Porte 2 (pequena): 6-15 lançamentos
  Porte 3 (média):   16-40 lançamentos
  Porte 4 (grande):  41-80 lançamentos
  Porte 5 (major):   81+ lançamentos

Distribuição de portes (limites absolutos):
  Porte 1:  288 distribuidoras ( 79.6%)
  Porte 2:   18 distribuidoras (  5.0%)
  Porte 3:   22 distribuidoras (  6.1%)
  Porte 4:   15 distribuidoras (  4.1%)
  Porte 5:   19 distribuidoras (  5.2%)

✓ Porte das distribuidoras calculado com limites absolutos


In [12]:
# 4.5. Análise de Majors - Top Distribuidoras por Público/Renda

print("\n" + "="*60)
print("ANÁLISE DE MAJORS (2012-2025) - POR PÚBLICO")
print("="*60)

# filtro de período
df_lanc_filtrado = df_lanc_filtrado[df_lanc_filtrado['ano_lancamento'].between(2012, 2025)]


# Identificar coluna de público
publico_cols = [col for col in df_lanc_filtrado.columns if 'publico' in col.lower()]
renda_cols = [col for col in df_lanc_filtrado.columns if 'renda' in col.lower()]

print(f"Colunas de público disponíveis: {publico_cols}")
print(f"Colunas de renda disponíveis: {renda_cols}")

# Usar público como métrica principal
if publico_cols:
    metrica_col = publico_cols[0]
    metrica_nome = "Público"
elif renda_cols:
    metrica_col = renda_cols[0]
    metrica_nome = "Renda"
else:
    print("\n⚠️  ATENÇÃO: Não foi possível identificar coluna de público ou renda!")
    print("Usando número de lançamentos como fallback.\n")
    metrica_col = 'cpb_roe'
    metrica_nome = "Lançamentos"

# Garantir que a métrica seja numérica
df_lanc_filtrado[metrica_col] = pd.to_numeric(df_lanc_filtrado[metrica_col], errors='coerce')

# Público/Renda total por distribuidora e ano
if metrica_nome == "Lançamentos":
    dist_por_ano = df_lanc_filtrado.groupby(['razao_social_dist_norm', 'ano_lancamento']).agg({
        metrica_col: 'nunique'
    }).reset_index()
else:
    dist_por_ano = df_lanc_filtrado.groupby(['razao_social_dist_norm', 'ano_lancamento']).agg({
        metrica_col: 'sum'
    }).reset_index()

dist_por_ano.columns = ['distribuidora', 'ano', 'metrica']

# Top 10 por ano
print(f"\nTop 10 distribuidoras por {metrica_nome} por ano:\n")
for ano in sorted(df_lanc_filtrado['ano_lancamento'].unique()):
    top10_ano = dist_por_ano[dist_por_ano['ano'] == ano].nlargest(10, 'metrica')
    print(f"\n{ano}:")
    for idx, row in top10_ano.iterrows():
        if metrica_nome == "Público":
            print(f"  {row['distribuidora'][:45]:45s} - {row['metrica']:>12,.0f} espectadores")
        elif metrica_nome == "Renda":
            print(f"  {row['distribuidora'][:45]:45s} - R$ {row['metrica']:>12,.2f}")
        else:
            print(f"  {row['distribuidora'][:45]:45s} - {row['metrica']:>12,.0f} lançamentos")

# Distribuidoras que aparecem consistentemente no top 10
anos_disponiveis = len(df_lanc_filtrado['ano_lancamento'].unique())
print(f"\n{'='*60}")
print(f"Distribuidoras no Top 10 em pelo menos 70% dos anos ({int(anos_disponiveis * 0.7)} de {anos_disponiveis} anos):\n")

# Contar em quantos anos cada distribuidora esteve no top 10
dist_top10_count = {}
for ano in df_lanc_filtrado['ano_lancamento'].unique():
    top10_ano = dist_por_ano[dist_por_ano['ano'] == ano].nlargest(10, 'metrica')['distribuidora'].tolist()
    for dist in top10_ano:
        dist_top10_count[dist] = dist_top10_count.get(dist, 0) + 1

# Filtrar as consistentes (>= 70% dos anos)
threshold = anos_disponiveis * 0.7
majors_candidatas = {dist: count for dist, count in dist_top10_count.items() if count >= threshold}

# Ordenar por frequência
majors_candidatas_sorted = sorted(majors_candidatas.items(), key=lambda x: x[1], reverse=True)

# Calcular público/renda total por distribuidora (2012+)
if metrica_nome == "Lançamentos":
    dist_total = df_lanc_filtrado.groupby('razao_social_dist_norm')[metrica_col].nunique().to_dict()
else:
    dist_total = df_lanc_filtrado.groupby('razao_social_dist_norm')[metrica_col].sum().to_dict()

for dist, anos_top10 in majors_candidatas_sorted:
    total = dist_total.get(dist, 0)
    if metrica_nome == "Público":
        print(f"  {dist[:40]:40s} - Top10 em {anos_top10:2d}/{anos_disponiveis} anos - Total: {total:>12,.0f} espectadores")
    elif metrica_nome == "Renda":
        print(f"  {dist[:40]:40s} - Top10 em {anos_top10:2d}/{anos_disponiveis} anos - Total: R$ {total:>12,.2f}")
    else:
        print(f"  {dist[:40]:40s} - Top10 em {anos_top10:2d}/{anos_disponiveis} anos - Total: {total:>12,.0f} lançamentos")

print(f"\n{'='*60}")
print(f"Total de candidatas a major: {len(majors_candidatas_sorted)}")
print(f"{'='*60}\n")

# Salvar lista para usar depois
lista_majors_empirica = [dist for dist, _ in majors_candidatas_sorted]


ANÁLISE DE MAJORS (2012-2025) - POR PÚBLICO
Colunas de público disponíveis: ['publico_total']
Colunas de renda disponíveis: ['renda_total']

Top 10 distribuidoras por Público por ano:


2012:
  fox film do brasil ltda                       -   22,617,257 espectadores
  columbia tristar filmes do brasil ltda        -   21,841,044 espectadores
  sm distribuidora de filmes ltda               -   21,137,430 espectadores
  paramount pictures brasil distribuidora de fi -   21,056,642 espectadores
  warner bros. (south) inc.                     -   18,607,019 espectadores
  the walt disney company (brasil) ltda.        -   17,369,978 espectadores
  wmix distribuidora ltda.                      -   11,172,175 espectadores
  freespirit distribuidora de filmes ltda.      -   10,341,270 espectadores
  playarte pictures entretenimentos ltda.       -    1,783,805 espectadores
  antonio fernandes filmes ltda                 -    1,683,239 espectadores

2013:
  the walt disney company (brasil) ltda.

In [13]:
# 4.6. Definir Majors Empiricamente + Mapeamento de Fusões

print("\nDefinindo majors empiricamente...\n")

# Mapeamento de fusões, aquisições e variações de nome
mapeamento_majors = {
    # Warner
    'warner': 'warner',
    'warner bros': 'warner',
    
    # Sony/Columbia
    'sony': 'sony',
    'columbia': 'sony',
    'columbia tristar': 'sony',
    'tristar': 'sony',
    
    # Disney (incluindo Fox pós-2019)
    'disney': 'disney',
    'walt disney': 'disney',
    'buena vista': 'disney',
    'fox': 'disney',  # Aquisição em 2019
    '20th century': 'disney',
    'twentieth century': 'disney',
    
    # Paramount
    'paramount': 'paramount',
    
    # Universal
    'universal': 'universal',
    'uip': 'universal',  # United International Pictures (parceria Universal+Paramount, encerrada)
}

def identificar_major_empirico(nome_dist):
    """Identifica major baseado em dados empíricos + mapeamento"""
    if pd.isna(nome_dist):
        return None
    
    nome_norm = normalizar_texto(nome_dist)
    if nome_norm is None:
        return None
    
    # Verificar mapeamento
    for chave, major in mapeamento_majors.items():
        if chave in nome_norm:
            return major
    
    return None

# Testar com as candidatas identificadas
print("Classificação das candidatas:\n")
for dist, _ in majors_candidatas_sorted:
    major = identificar_major_empirico(dist)
    total_pub = dist_total.get(dist, 0)
    if major:
        print(f"  ✓ MAJOR: {dist[:40]:40s} → {major.upper():10s} ({total_pub:>12,.0f} esp.)")
    else:
        print(f"  ✗ INDEP: {dist[:40]:40s} ({total_pub:>12,.0f} esp.)")

print("\n" + "="*60)


Definindo majors empiricamente...

Classificação das candidatas:

  ✓ MAJOR: warner bros. (south) inc.                → WARNER     ( 361,394,018 esp.)
  ✓ MAJOR: the walt disney company (brasil) ltda.   → DISNEY     ( 438,127,058 esp.)
  ✓ MAJOR: paramount pictures brasil distribuidora  → PARAMOUNT  ( 139,845,782 esp.)
  ✓ MAJOR: columbia tristar filmes do brasil ltda   → SONY       ( 248,373,622 esp.)
  ✗ INDEP: sm distribuidora de filmes ltda          ( 186,571,792 esp.)
  ✗ INDEP: wmix distribuidora ltda.                 (  62,547,907 esp.)
  ✗ INDEP: diamond films do brasil producao e distr (  27,148,270 esp.)
  ✗ INDEP: freespirit distribuidora de filmes ltda. (  77,957,582 esp.)



## 5. Construção do DataFrame Base de Obras

### 5.1. Agregar Produtoras por Obra

In [14]:
print("\nAgregando produtoras por obra...\n")

# Converter para Pandas
df_prod_cpb = df_produtores_cpb.to_pandas()
df_prod_roe = df_produtores_roe.to_pandas()

# Identificar colunas corretas (variam conforme dataset)
cpb_cols = [col for col in df_prod_cpb.columns if 'cpb' in col.lower()]
roe_cols = [col for col in df_prod_roe.columns if 'roe' in col.lower()]
prod_col_cpb = [col for col in df_prod_cpb.columns if 'produtor' in col.lower()][0]
prod_col_roe = [col for col in df_prod_roe.columns if 'produtor' in col.lower()][0]

print(f"CPB: coluna obra = {cpb_cols[0]}, coluna produtor = {prod_col_cpb}")
print(f"ROE: coluna obra = {roe_cols[0]}, coluna produtor = {prod_col_roe}")

# Normalizar nomes de produtoras
df_prod_cpb['produtor_norm'] = df_prod_cpb[prod_col_cpb].apply(normalizar_texto)
df_prod_roe['produtor_norm'] = df_prod_roe[prod_col_roe].apply(normalizar_texto)

# Criar coluna CPB_ROE
df_prod_cpb['cpb_roe'] = df_prod_cpb[cpb_cols[0]]
df_prod_roe['cpb_roe'] = df_prod_roe[roe_cols[0]]

# Remover nulos
df_prod_cpb = df_prod_cpb[df_prod_cpb['produtor_norm'].notna()]
df_prod_roe = df_prod_roe[df_prod_roe['produtor_norm'].notna()]

# Agregar produtoras em listas por obra
produtoras_cpb = df_prod_cpb.groupby('cpb_roe')['produtor_norm'].apply(list).reset_index()
produtoras_cpb.columns = ['cpb_roe', 'produtoras_lista']

produtoras_roe = df_prod_roe.groupby('cpb_roe')['produtor_norm'].apply(list).reset_index()
produtoras_roe.columns = ['cpb_roe', 'produtoras_lista']

# Unir CPB e ROE
df_obras_produtoras = pd.concat([produtoras_cpb, produtoras_roe], ignore_index=True)

# Remover duplicatas na lista
df_obras_produtoras['produtoras_lista'] = df_obras_produtoras['produtoras_lista'].apply(
    lambda x: list(set(x)) if isinstance(x, list) else x
)

print(f"\nObras com produtoras: {len(df_obras_produtoras)}")
print(f"Exemplo: {df_obras_produtoras.head(2).to_dict('records')}")
print("\n✓ Produtoras agregadas")


Agregando produtoras por obra...

CPB: coluna obra = cpb, coluna produtor = produtor
ROE: coluna obra = roe, coluna produtor = produtor

Obras com produtoras: 87247
Exemplo: [{'cpb_roe': 'B0200000400000', 'produtoras_lista': ['cinema brasil digital - escritorio de planej. empr. aud. ltda', 'distribuidora de filmes s/a riofilme', 'brasil filmes ltda']}, {'cpb_roe': 'B0200000600000', 'produtoras_lista': ['conceito eventos e promocoes ltda']}]

✓ Produtoras agregadas


### 5.2. Agregar Distribuidoras por Obra

In [15]:
print("\nAgregando distribuidoras por obra...\n")

# Remover nulos
df_lanc_limpo = df_lanc[df_lanc['razao_social_dist_norm'].notna()].copy()

# Agregar distribuidoras em listas por obra
distribuidoras_obra = df_lanc_limpo.groupby('cpb_roe')['razao_social_dist_norm'].apply(list).reset_index()
distribuidoras_obra.columns = ['cpb_roe', 'distribuidoras_lista']

# Remover duplicatas na lista
distribuidoras_obra['distribuidoras_lista'] = distribuidoras_obra['distribuidoras_lista'].apply(
    lambda x: list(set(x)) if isinstance(x, list) else x
)

print(f"\nObras com distribuidoras: {len(distribuidoras_obra)}")
print(f"Exemplo: {distribuidoras_obra.head(2).to_dict('records')}")
print("\n✓ Distribuidoras agregadas")


Agregando distribuidoras por obra...


Obras com distribuidoras: 6621
Exemplo: [{'cpb_roe': 'B0300040500000', 'distribuidoras_lista': ['vitrine filmes ltda']}, {'cpb_roe': 'B0300046700000', 'distribuidoras_lista': ['trinca filmes ltda.']}]

✓ Distribuidoras agregadas


### 5.3. Unir Produtoras e Distribuidoras

In [16]:
print("\nUnindo produtoras e distribuidoras...\n")

# Merge outer para não perder obras
df_obras = df_obras_produtoras.merge(
    distribuidoras_obra,
    on='cpb_roe',
    how='outer'
)

print(f"Total de obras únicas: {len(df_obras)}")
print(f"Obras com produtoras: {df_obras['produtoras_lista'].notna().sum()}")
print(f"Obras com distribuidoras: {df_obras['distribuidoras_lista'].notna().sum()}")
print(f"Obras com ambos: {(df_obras['produtoras_lista'].notna() & df_obras['distribuidoras_lista'].notna()).sum()}")

print("\n✓ Base de obras unificada")


Unindo produtoras e distribuidoras...

Total de obras únicas: 87247
Obras com produtoras: 87247
Obras com distribuidoras: 6621
Obras com ambos: 6621

✓ Base de obras unificada


In [17]:
# Filtrar apenas obras que existem na base de sessões
print("\n[FILTRO] Mantendo apenas obras da base de sessões...")
antes = len(df_obras)
df_obras = df_obras[df_obras['cpb_roe'].isin(obras_validas)]
depois = len(df_obras)

print(f"  Obras antes do filtro: {antes:,}")
print(f"  Obras após filtro: {depois:,}")
print(f"  Obras removidas: {antes - depois:,}")
print("\n✓ Base filtrada")


[FILTRO] Mantendo apenas obras da base de sessões...
  Obras antes do filtro: 87,247
  Obras após filtro: 2,189
  Obras removidas: 85,058

✓ Base filtrada


## 6. Enriquecimento de Produtoras

In [18]:
print("\nEnriquecendo informações de produtoras...\n")

# Converter datasets para Pandas
df_agentes_pd = df_agentes.to_pandas()
df_produtoras_indep_pd = df_produtoras_indep.to_pandas()

# Identificar colunas
razao_col_agentes = [col for col in df_agentes_pd.columns if 'razao' in col.lower()][0]
razao_col_indep = [col for col in df_produtoras_indep_pd.columns if 'razao' in col.lower()][0]
nivel_col = [col for col in df_produtoras_indep_pd.columns if 'nivel' in col.lower()][0]
brasileiro_col = [col for col in df_agentes_pd.columns if 'brasileiro' in col.lower()][0]

print(f"Colunas identificadas:")
print(f"  Agentes - Razão Social: {razao_col_agentes}")
print(f"  Agentes - Brasileiro Independente: {brasileiro_col}")
print(f"  Produtoras Indep - Razão Social: {razao_col_indep}")
print(f"  Produtoras Indep - Nível: {nivel_col}")

# Normalizar nomes
df_agentes_pd['razao_norm'] = df_agentes_pd[razao_col_agentes].apply(normalizar_texto)
df_produtoras_indep_pd['razao_norm'] = df_produtoras_indep_pd[razao_col_indep].apply(normalizar_texto)

# Criar dicionários de lookup
dict_brasileiro_indep = df_agentes_pd.set_index('razao_norm')[brasileiro_col].to_dict()
dict_nivel = df_produtoras_indep_pd.set_index('razao_norm')[nivel_col].to_dict()

print(f"\nProdutoras no dict de nível: {len(dict_nivel)}")
print(f"Agentes no dict brasileiro_indep: {len(dict_brasileiro_indep)}")

print("\n✓ Dicionários de enriquecimento criados")


Enriquecendo informações de produtoras...

Colunas identificadas:
  Agentes - Razão Social: razao_social
  Agentes - Brasileiro Independente: brasileiro_independente
  Produtoras Indep - Razão Social: razao_social
  Produtoras Indep - Nível: classificacao_nivel_produtora

Produtoras no dict de nível: 11977
Agentes no dict brasileiro_indep: 18794

✓ Dicionários de enriquecimento criados


## 7. Criação das Features Derivadas

In [19]:
print("\nCriando features derivadas...\n")

def calcular_features_produtoras(lista_produtoras):
    """Calcula features de produtoras para uma obra"""
    if not isinstance(lista_produtoras, list) or len(lista_produtoras) == 0:
        return {
            'num_produtoras': 0,
            'num_produtoras_brasileiras': 0,
            'produtora_nivel_medio': None,
            'produtora_nivel_min': None,
            'produtora_nivel_max': None,
            'produtora_independente': None,
            'tem_produtora_brasileira': False
        }
    
    # Buscar níveis das produtoras BRASILEIRAS
    niveis = []
    produtoras_brasileiras = []
    
    for p in lista_produtoras:
        nivel = dict_nivel.get(p)
        if nivel is not None and pd.notna(nivel):
            niveis.append(nivel)
            produtoras_brasileiras.append(p)
    
    num_total = len(lista_produtoras)
    num_brasileiras = len(produtoras_brasileiras)
    tem_brasileira = num_brasileiras > 0
    
    # Se não tem nenhuma produtora brasileira com nível identificado
    if len(niveis) == 0:
        return {
            'num_produtoras': num_total,
            'num_produtoras_brasileiras': num_brasileiras,
            'produtora_nivel_medio': None,
            'produtora_nivel_min': None,
            'produtora_nivel_max': None,
            'produtora_independente': None,
            'tem_produtora_brasileira': tem_brasileira
        }
    
    nivel_min = min(niveis)
    nivel_max = max(niveis)
    nivel_medio = np.mean(niveis)
    
    # Obra é independente se TEM pelo menos uma produtora brasileira independente (nível <= 2)
    # Não importa se tem outras estrangeiras ou de nível alto
    is_independente = nivel_min <= 2
    
    return {
        'num_produtoras': num_total,
        'num_produtoras_brasileiras': num_brasileiras,
        'produtora_nivel_medio': nivel_medio,
        'produtora_nivel_min': nivel_min,
        'produtora_nivel_max': nivel_max,
        'produtora_independente': is_independente,
        'tem_produtora_brasileira': tem_brasileira
    }


def calcular_features_distribuidoras(lista_distribuidoras):
    """Calcula features de distribuidoras para uma obra"""
    if not isinstance(lista_distribuidoras, list) or len(lista_distribuidoras) == 0:
        return {
            'num_distribuidoras': 0,
            'distribuidora_porte': None,
            'distribuidora_independente': None,
            'distribuidora_major': None,
            'distribuidora_major_nome': None
        }
    
    # Buscar portes
    portes = []
    for d in lista_distribuidoras:
        porte_info = lancamentos_por_dist[lancamentos_por_dist['distribuidora_norm'] == d]
        if len(porte_info) > 0:
            portes.append(porte_info.iloc[0]['porte'])
    
    # Identificar majors (retorna nome da major ou None)
    majors_encontradas = []
    for d in lista_distribuidoras:
        major = identificar_major_empirico(d)
        if major:
            majors_encontradas.append(major)
    
    tem_major = len(majors_encontradas) > 0
    nome_major = majors_encontradas[0] if majors_encontradas else None  # Pega primeira major se houver
    
    if len(portes) == 0:
        return {
            'num_distribuidoras': len(lista_distribuidoras),
            'distribuidora_porte': None,
            'distribuidora_independente': None,
            'distribuidora_major': tem_major,
            'distribuidora_major_nome': nome_major
        }
    
    porte_medio = int(np.round(np.mean(portes)))

    # Flag: distribuidora foi informada?
    tem_info_distribuidor = isinstance(lista_distribuidoras, list) and len(lista_distribuidoras) > 0
    
    if not tem_info_distribuidor:
        return {
            'num_distribuidoras': 0,
            'distribuidora_porte': None,
            'distribuidora_independente': None,
            'distribuidora_major': None,
            'distribuidora_major_nome': None,
            'distribuidora_informada': False  # <- NOVA FLAG
        }
    
    return {
        'num_distribuidoras': len(lista_distribuidoras),
        'distribuidora_porte': porte_medio,
        'distribuidora_independente': porte_medio <= 2,
        'distribuidora_major': tem_major,
        'distribuidora_major_nome': nome_major,
        'distribuidora_informada': True  # <- NOVA FLAG
    }


# Aplicar features
print("Calculando features de produtoras...")
features_prod = df_obras['produtoras_lista'].apply(calcular_features_produtoras)
df_features_prod = pd.DataFrame(features_prod.tolist())

print("Calculando features de distribuidoras...")
features_dist = df_obras['distribuidoras_lista'].apply(calcular_features_distribuidoras)
df_features_dist = pd.DataFrame(features_dist.tolist())

# Unir ao dataframe principal
df_obras_enriquecido = pd.concat([
    df_obras.reset_index(drop=True),
    df_features_prod.reset_index(drop=True),
    df_features_dist.reset_index(drop=True)
], axis=1)

print(f"\n✓ Features criadas")
print(f"Shape final: {df_obras_enriquecido.shape}")
print(f"\nColunas finais: {df_obras_enriquecido.columns.tolist()}")


Criando features derivadas...

Calculando features de produtoras...
Calculando features de distribuidoras...

✓ Features criadas
Shape final: (2189, 16)

Colunas finais: ['cpb_roe', 'produtoras_lista', 'distribuidoras_lista', 'num_produtoras', 'num_produtoras_brasileiras', 'produtora_nivel_medio', 'produtora_nivel_min', 'produtora_nivel_max', 'produtora_independente', 'tem_produtora_brasileira', 'num_distribuidoras', 'distribuidora_porte', 'distribuidora_independente', 'distribuidora_major', 'distribuidora_major_nome', 'distribuidora_informada']


## 8. Estatísticas e Validação

In [20]:
print("\n" + "=" * 60)
print("ESTATÍSTICAS DO ENRIQUECIMENTO")
print("=" * 60)

total_obras = len(df_obras_enriquecido)

print(f"\n📊 COBERTURA GERAL")
print(f"  Total de obras: {total_obras:,}")
print(f"  Obras com produtoras: {df_obras_enriquecido['produtoras_lista'].notna().sum():,} ({df_obras_enriquecido['produtoras_lista'].notna().sum()/total_obras*100:.1f}%)")
print(f"  Obras com distribuidoras: {df_obras_enriquecido['distribuidoras_lista'].notna().sum():,} ({df_obras_enriquecido['distribuidoras_lista'].notna().sum()/total_obras*100:.1f}%)")

print(f"\n📊 PRODUTORAS")
obras_com_nivel = df_obras_enriquecido['produtora_nivel_min'].notna().sum()
obras_com_br = df_obras_enriquecido['tem_produtora_brasileira'].sum()
obras_independentes = df_obras_enriquecido['produtora_independente'].sum()

print(f"  Obras com produtora brasileira identificada: {obras_com_br:,} ({obras_com_br/total_obras*100:.1f}%)")
print(f"  Obras com nível identificado: {obras_com_nivel:,} ({obras_com_nivel/total_obras*100:.1f}%)")
print(f"  Obras sem nível: {total_obras - obras_com_nivel:,} ({(total_obras - obras_com_nivel)/total_obras*100:.1f}%)")
print(f"  Obras com produtora independente (nível ≤2): {obras_independentes:,} ({obras_independentes/total_obras*100:.1f}%)")

if obras_com_nivel > 0:
    print(f"\n  Distribuição de níveis (min) - APENAS PRODUTORAS BRASILEIRAS:")
    print(df_obras_enriquecido['produtora_nivel_min'].value_counts().sort_index())
    
    print(f"\n  Estatísticas de num_produtoras (total):")
    print(df_obras_enriquecido['num_produtoras'].describe())
    
    print(f"\n  Estatísticas de num_produtoras_brasileiras:")
    print(df_obras_enriquecido['num_produtoras_brasileiras'].describe())

print(f"\n📊 DISTRIBUIDORAS")
obras_com_porte = df_obras_enriquecido['distribuidora_porte'].notna().sum()
obras_com_major = df_obras_enriquecido['distribuidora_major'].sum()
obras_dist_independente = (df_obras_enriquecido['distribuidora_independente'] == True).sum()

print(f"  Obras com porte identificado: {obras_com_porte:,} ({obras_com_porte/total_obras*100:.1f}%)")
print(f"  Obras sem porte: {total_obras - obras_com_porte:,} ({(total_obras - obras_com_porte)/total_obras*100:.1f}%)")
print(f"  Obras com major: {obras_com_major:,} ({obras_com_major/total_obras*100:.1f}%)")
print(f"  Obras com distribuidora independente (porte ≤2): {obras_dist_independente:,} ({obras_dist_independente/total_obras*100:.1f}%)")
print(f"  Obras com distribuidor informado: {df_obras_enriquecido['distribuidora_informada'].sum():,} ({df_obras_enriquecido['distribuidora_informada'].sum()/total_obras*100:.1f}%)")

if obras_com_porte > 0:
    print(f"\n  Distribuição de portes:")
    print(df_obras_enriquecido['distribuidora_porte'].value_counts().sort_index())
    
    print(f"\n  Distribuição de majors (por nome):")
    print(df_obras_enriquecido['distribuidora_major_nome'].value_counts())
    
    print(f"\n  Estatísticas de num_distribuidoras:")
    print(df_obras_enriquecido['num_distribuidoras'].describe())

print(f"\n📊 CRUZAMENTOS INTERESSANTES")
# Obras brasileiras independentes
obras_br_indep = (df_obras_enriquecido['produtora_independente'] == True).sum()
print(f"  Obras com produtora BR independente: {obras_br_indep:,}")

# Obras com major + produtora independente
obras_major_indep = ((df_obras_enriquecido['distribuidora_major'] == True) & 
                     (df_obras_enriquecido['produtora_independente'] == True)).sum()
print(f"  Obras independentes distribuídas por major: {obras_major_indep:,}")

# Obras 100% brasileiras (só produtoras BR, sem estrangeiras)
obras_100_br = (df_obras_enriquecido['num_produtoras'] == df_obras_enriquecido['num_produtoras_brasileiras']).sum()
print(f"  Obras 100% brasileiras (só prod. BR): {obras_100_br:,} ({obras_100_br/total_obras*100:.1f}%)")

# Coproduções (tem BR + estrangeiras)
coprod = ((df_obras_enriquecido['tem_produtora_brasileira'] == True) & 
          (df_obras_enriquecido['num_produtoras'] > df_obras_enriquecido['num_produtoras_brasileiras'])).sum()
print(f"  Coproduções (BR + estrangeira): {coprod:,} ({coprod/total_obras*100:.1f}%)")

print("\n" + "=" * 60)


ESTATÍSTICAS DO ENRIQUECIMENTO

📊 COBERTURA GERAL
  Total de obras: 2,189
  Obras com produtoras: 2,189 (100.0%)
  Obras com distribuidoras: 1,710 (78.1%)

📊 PRODUTORAS
  Obras com produtora brasileira identificada: 775 (35.4%)
  Obras com nível identificado: 775 (35.4%)
  Obras sem nível: 1,414 (64.6%)
  Obras com produtora independente (nível ≤2): 376 (17.2%)

  Distribuição de níveis (min) - APENAS PRODUTORAS BRASILEIRAS:
produtora_nivel_min
1.0    252
2.0    124
3.0    117
4.0    149
5.0    133
Name: count, dtype: int64

  Estatísticas de num_produtoras (total):
count    2189.000000
mean        2.028780
std         1.417633
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max        14.000000
Name: num_produtoras, dtype: float64

  Estatísticas de num_produtoras_brasileiras:
count    2189.000000
mean        0.476016
std         0.737476
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         5.000000
Name: 

## 9. Salvar Outputs

### 9.1. Salvar df_obras_enriquecido.parquet

In [21]:
# Caminho correto: ../bases/df_obras_enriquecido.parquet
output_path = Path('..') / 'bases' / 'df_obras_enriquecido.parquet'

print(f"\nSalvando {output_path}...")

# Converter para Polars para salvar com compressão
df_obras_enriquecido_pl = pl.from_pandas(df_obras_enriquecido)

df_obras_enriquecido_pl.write_parquet(
    output_path,
    compression='snappy'
)

file_size = output_path.stat().st_size / (1024 * 1024)
print(f"✓ Arquivo salvo: {file_size:.2f} MB")
print(f"  Linhas: {len(df_obras_enriquecido):,}")
print(f"  Colunas: {len(df_obras_enriquecido.columns)}")


Salvando ..\bases\df_obras_enriquecido.parquet...
✓ Arquivo salvo: 0.09 MB
  Linhas: 2,189
  Colunas: 16


In [22]:
# 9.1b. Salvar df_obras_enriquecido.xlsx

output_xlsx = Path('..') / 'bases' / 'df_obras_enriquecido.xlsx'

print(f"\nSalvando {output_xlsx}...")

# Converter listas para string para compatibilidade com Excel
df_excel = df_obras_enriquecido.copy()
df_excel['produtoras_lista'] = df_excel['produtoras_lista'].apply(
    lambda x: '; '.join(x) if isinstance(x, list) else ''
)
df_excel['distribuidoras_lista'] = df_excel['distribuidoras_lista'].apply(
    lambda x: '; '.join(x) if isinstance(x, list) else ''
)

# Salvar
df_excel.to_excel(output_xlsx, index=False, engine='openpyxl')

xlsx_size = output_xlsx.stat().st_size / (1024 * 1024)
print(f"✓ Excel salvo: {xlsx_size:.2f} MB")



Salvando ..\bases\df_obras_enriquecido.xlsx...
✓ Excel salvo: 0.17 MB


### 9.2. Criar Dicionário de Dados

In [23]:
dicionario = {
    "metadata": {
        "nome_arquivo": "df_obras_enriquecido.parquet",
        "data_criacao": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "total_obras": len(df_obras_enriquecido),
        "periodo_distribuidoras": "2012-presente",
        "fontes": [
            "Agentes Econômicos Regulares (ANCINE)",
            "Produtoras Independentes Regulares (ANCINE)",
            "Produtores de Obras Brasileiras (ANCINE)",
            "Produtores de Obras Estrangeiras (ANCINE)",
            "Lançamentos Comerciais por Distribuidoras (ANCINE)"
        ]
    },
   "colunas": [
        {
            "nome": "cpb_roe",
            "tipo": "string",
            "descricao": "Identificador único da obra (CPB para brasileiras, ROE para estrangeiras)",
            "origem": "Conjuntos de produtores e lançamentos",
            "exemplo": "B1234567890000"
        },
        {
            "nome": "produtoras_lista",
            "tipo": "list[string]",
            "descricao": "Lista normalizada de produtoras da obra",
            "origem": "Produtores CPB e ROE",
            "transformacao": "Normalização: lowercase, sem acentos, sem duplicatas"
        },
        {
            "nome": "distribuidoras_lista",
            "tipo": "list[string]",
            "descricao": "Lista normalizada de distribuidoras da obra",
            "origem": "Lançamentos Comerciais",
            "transformacao": "Normalização + regras de limpeza (ROE E1300000100000 excluído, distribuidoras problemáticas tratadas)"
        },
        {
            "nome": "num_produtoras",
            "tipo": "int",
            "descricao": "Número total de produtoras da obra (brasileiras + estrangeiras)",
            "origem": "Derivado de produtoras_lista",
            "valores_possiveis": "0 a N"
        },
        {
            "nome": "num_produtoras_brasileiras",
            "tipo": "int",
            "descricao": "Número de produtoras brasileiras com nível identificado",
            "origem": "Derivado do cruzamento com Produtoras Independentes Regulares",
            "valores_possiveis": "0 a N",
            "observacao": "Apenas produtoras brasileiras regulares têm nível"
        },
        {
            "nome": "tem_produtora_brasileira",
            "tipo": "bool",
            "descricao": "Indica se obra tem pelo menos uma produtora brasileira identificada",
            "origem": "Derivado de num_produtoras_brasileiras",
            "valores_possiveis": "True ou False"
        },
        {
            "nome": "produtora_nivel_medio",
            "tipo": "float",
            "descricao": "Nível médio das produtoras BRASILEIRAS (1-5, quando identificado)",
            "origem": "Produtoras Independentes Regulares (campo Nível)",
            "valores_possiveis": "1.0 a 5.0 ou null",
            "observacao": "Considera apenas produtoras brasileiras regulares. Estrangeiras não têm nível."
        },
        {
            "nome": "produtora_nivel_min",
            "tipo": "int",
            "descricao": "Nível mínimo das produtoras BRASILEIRAS da obra",
            "origem": "Produtoras Independentes Regulares (campo Nível)",
            "valores_possiveis": "1 a 5 ou null",
            "observacao": "Considera apenas produtoras brasileiras"
        },
        {
            "nome": "produtora_nivel_max",
            "tipo": "int",
            "descricao": "Nível máximo das produtoras BRASILEIRAS da obra",
            "origem": "Produtoras Independentes Regulares (campo Nível)",
            "valores_possiveis": "1 a 5 ou null",
            "observacao": "Considera apenas produtoras brasileiras"
        },
        {
            "nome": "produtora_independente",
            "tipo": "bool",
            "descricao": "Indica se obra tem pelo menos uma produtora brasileira independente (nível <= 2)",
            "origem": "Derivado de produtora_nivel_min",
            "regra": "True se tem ao menos uma produtora brasileira com nível <= 2",
            "valores_possiveis": "True, False ou null",
            "observacao": "Null quando não há produtoras brasileiras com nível identificado"
        },
        
        
        {
            "nome": "distribuidora_major",
            "tipo": "bool",
            "descricao": "Indica se obra tem distribuidora major",
            "origem": "Identificação empírica via análise de público 2012-presente",
            "regra": "True se nome normalizado corresponde a Warner, Sony/Columbia, Disney/Fox, Paramount ou Universal",
            "valores_possiveis": "True, False ou null"
        },
        {
            "nome": "distribuidora_major_nome",
            "tipo": "string",
            "descricao": "Nome da major identificada (se houver)",
            "origem": "Mapeamento de fusões e variações de nome",
            "valores_possiveis": "'warner', 'sony', 'disney', 'paramount', 'universal' ou null",
            "observacao": "Em co-distribuição, retorna primeira major encontrada"
        },
        {
            "nome": "distribuidora_informada",
            "tipo": "bool",
            "descricao": "Indica se há informação de distribuidor para a obra",
            "origem": "Derivado de distribuidoras_lista",
            "regra": "False quando distribuidoras_lista é vazia/null",
            "valores_possiveis": "True ou False",
            "observacao": "21.9% das obras não têm distribuidor informado devido a gap nos dados ANCINE. Destes, 6.5% têm padrão de distribuição comercial ampla."
        }
    ],

    "regras_limpeza": [
        "ROE E1300000100000 excluído completamente",
        "KOMPANHIA TEATRO MULTIMÃDIA DE SÃO PAULO: removida se co-distribuidora, mantida se única",
        "CINE CAXAMBU LTDA: removida se co-distribuidora, mantida se única",
        "Período de análise de distribuidoras: 2012 até data mais recente",
        "Duplicatas removidas em listas de produtoras e distribuidoras"
    ],
    "missings": {
        "produtoras": f"{(df_obras_enriquecido['produtoras_lista'].isna().sum() / len(df_obras_enriquecido) * 100):.1f}%",
        "distribuidoras": f"{(df_obras_enriquecido['distribuidoras_lista'].isna().sum() / len(df_obras_enriquecido) * 100):.1f}%",
        "nivel_produtoras": f"{(df_obras_enriquecido['produtora_nivel_min'].isna().sum() / len(df_obras_enriquecido) * 100):.1f}%",
        "porte_distribuidoras": f"{(df_obras_enriquecido['distribuidora_porte'].isna().sum() / len(df_obras_enriquecido) * 100):.1f}%"
    }
}

dict_path = Path('dicionario_df_obras_enriquecido.json')
with open(dict_path, 'w', encoding='utf-8') as f:
    json.dump(dicionario, f, ensure_ascii=False, indent=2)

print(f"\n✓ Dicionário salvo: {dict_path}")


✓ Dicionário salvo: dicionario_df_obras_enriquecido.json


### 9.3. Relatório Final em Markdown

In [24]:
relatorio = f"""
# Relatório de Enriquecimento de Obras
**Data:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## Resumo Executivo

- **Total de obras processadas:** {len(df_obras_enriquecido):,}
- **Obras com produtoras:** {df_obras_enriquecido['produtoras_lista'].notna().sum():,} ({df_obras_enriquecido['produtoras_lista'].notna().sum()/len(df_obras_enriquecido)*100:.1f}%)
- **Obras com distribuidoras:** {df_obras_enriquecido['distribuidoras_lista'].notna().sum():,} ({df_obras_enriquecido['distribuidoras_lista'].notna().sum()/len(df_obras_enriquecido)*100:.1f}%)

## Cobertura de Enriquecimento

### Produtoras
- **Obras com nível identificado:** {df_obras_enriquecido['produtora_nivel_min'].notna().sum():,} ({df_obras_enriquecido['produtora_nivel_min'].notna().sum()/len(df_obras_enriquecido)*100:.1f}%)
- **Obras sem nível:** {df_obras_enriquecido['produtora_nivel_min'].isna().sum():,} ({df_obras_enriquecido['produtora_nivel_min'].isna().sum()/len(df_obras_enriquecido)*100:.1f}%)

**Distribuição de Níveis (mínimo):**
{df_obras_enriquecido['produtora_nivel_min'].value_counts().sort_index().to_string()}

### Distribuidoras
- **Obras com porte identificado:** {df_obras_enriquecido['distribuidora_porte'].notna().sum():,} ({df_obras_enriquecido['distribuidora_porte'].notna().sum()/len(df_obras_enriquecido)*100:.1f}%)
- **Obras sem porte:** {df_obras_enriquecido['distribuidora_porte'].isna().sum():,} ({df_obras_enriquecido['distribuidora_porte'].isna().sum()/len(df_obras_enriquecido)*100:.1f}%)
- **Obras com major:** {df_obras_enriquecido['distribuidora_major'].sum():,} ({df_obras_enriquecido['distribuidora_major'].sum()/len(df_obras_enriquecido)*100:.1f}%)

**Distribuição de Portes:**
{df_obras_enriquecido['distribuidora_porte'].value_counts().sort_index().to_string()}

## Estatísticas das Features

### Número de Produtoras por Obra
{df_obras_enriquecido['num_produtoras'].describe().to_string()}

### Número de Distribuidoras por Obra
{df_obras_enriquecido['num_distribuidoras'].describe().to_string()}

## Outputs Gerados

1. **df_obras_enriquecido.parquet** - {file_size:.2f} MB
2. **df_obras_enriquecido.xlsx**
3. **dicionario_df_obras_enriquecido.json**
4. **relatorio_enriquecimento.md** (este arquivo)

## Próximos Passos

Esta base será utilizada nos Notebooks 1 e 2 através de JOIN com `df_sessoes_limpo.parquet` usando a chave `CPB_ROE`.
"""

relatorio_path = Path('relatorio_enriquecimento.md')
with open(relatorio_path, 'w', encoding='utf-8') as f:
    f.write(relatorio)

print(f"✓ Relatório salvo: {relatorio_path}")

✓ Relatório salvo: relatorio_enriquecimento.md


## 10. Finalização

In [25]:
print("\n" + "=" * 60)
print("✓ NOTEBOOK 0.5 CONCLUÍDO COM SUCESSO")
print("=" * 60)
print("\nArquivos gerados:")
print("  1. df_obras_enriquecido.parquet")
print("  2. dicionario_df_obras_enriquecido.json")
print("  3. relatorio_enriquecimento.md")
print("  4. df_obras_enriquecido.xlsx")
print("\nPróximo passo: Usar df_obras_enriquecido em JOIN com df_sessoes_limpo")
print("=" * 60)


✓ NOTEBOOK 0.5 CONCLUÍDO COM SUCESSO

Arquivos gerados:
  1. df_obras_enriquecido.parquet
  2. dicionario_df_obras_enriquecido.json
  3. relatorio_enriquecimento.md
  4. df_obras_enriquecido.xlsx

Próximo passo: Usar df_obras_enriquecido em JOIN com df_sessoes_limpo


# Análise extra - Obras sem distribuidora

In [26]:
# Características das 479 obras sem distribuidora
sem_dist = df_obras_enriquecido[df_obras_enriquecido['distribuidoras_lista'].isna()]

print(f"Total sem distribuidora: {len(sem_dist)}")
print(f"\nBrasileiras (CPB): {sem_dist['cpb_roe'].str.startswith('B', na=False).sum()}")
print(f"Estrangeiras (ROE): {sem_dist['cpb_roe'].str.startswith('E', na=False).sum()}")

print(f"\nCom produtora brasileira identificada: {sem_dist['tem_produtora_brasileira'].sum()}")
print(f"Independentes brasileiras: {sem_dist['produtora_independente'].sum()}")

# Ver alguns exemplos
print("\nExemplos de CPBs sem distribuidora:")
print(sem_dist[sem_dist['cpb_roe'].str.startswith('B', na=False)]['cpb_roe'].head(10).tolist())

Total sem distribuidora: 479

Brasileiras (CPB): 259
Estrangeiras (ROE): 220

Com produtora brasileira identificada: 170
Independentes brasileiras: 106

Exemplos de CPBs sem distribuidora:
['B0200001000000', 'B0300002700000', 'B0300002800000', 'B0300030800000', 'B0300047100000', 'B0300048000000', 'B0400059500000', 'B0400066700000', 'B0400067000000', 'B0400067100000']


In [27]:
# ============================================================
# ANÁLISE: Top 10 Obras com Maior Distribuição Sem Distribuidor
# ============================================================

print("\n" + "=" * 120)
print("ANÁLISE: OBRAS SEM DISTRIBUIDOR REGISTRADO")
print("=" * 120)

# Identificar obras sem distribuidora
sem_dist = df_obras_enriquecido[df_obras_enriquecido['distribuidoras_lista'].isna()]

# Analisar cada obra sem distribuidora
caracteristicas = []

for cpb in sem_dist['cpb_roe']:
    # Usar .filter() para Polars
    sessoes_cpb = df_sessoes.filter(pl.col('CPB_ROE') == cpb)
    
    if len(sessoes_cpb) > 0:
        caracteristicas.append({
            'cpb_roe': cpb,
            'total_sessoes': len(sessoes_cpb),
            'complexos': sessoes_cpb['NOME_COMPLEXO'].n_unique(),
            'salas': sessoes_cpb['REGISTRO_SALA'].n_unique(),
            'publico_total': sessoes_cpb['PUBLICO'].sum(),
            'primeira_sessao': sessoes_cpb['DATA_EXIBICAO'].min(),
            'ultima_sessao': sessoes_cpb['DATA_EXIBICAO'].max(),
            'titulo': sessoes_cpb['TITULO_OBRA'][0]
        })

df_sem_dist_analise = pd.DataFrame(caracteristicas)

# Calcular duração
df_sem_dist_analise['duracao_dias'] = (
    df_sem_dist_analise['ultima_sessao'] - df_sem_dist_analise['primeira_sessao']
).dt.days

print(f"\n📊 Total de obras sem distribuidor: {len(df_sem_dist_analise)}")
print(f"   Brasileiras (CPB): {sum(1 for cpb in df_sem_dist_analise['cpb_roe'] if cpb.startswith('B'))}")
print(f"   Estrangeiras (ROE): {sum(1 for cpb in df_sem_dist_analise['cpb_roe'] if cpb.startswith('E'))}")

# Classificar por padrão de exibição
df_sem_dist_analise['categoria'] = 'indefinido'

# Usar máscara booleana corretamente
mask_especial = (df_sem_dist_analise['total_sessoes'] <= 5) & (df_sem_dist_analise['complexos'] <= 2)
df_sem_dist_analise.loc[mask_especial, 'categoria'] = 'sessao_especial'

mask_comercial = (df_sem_dist_analise['total_sessoes'] > 20) | (df_sem_dist_analise['complexos'] > 5)
df_sem_dist_analise.loc[mask_comercial, 'categoria'] = 'distribuicao_comercial'

mask_festival = ((df_sem_dist_analise['total_sessoes'] >= 6) & 
                 (df_sem_dist_analise['total_sessoes'] <= 20) & 
                 (df_sem_dist_analise['complexos'] >= 2) & 
                 (df_sem_dist_analise['complexos'] <= 5))
df_sem_dist_analise.loc[mask_festival, 'categoria'] = 'mostra_festival'

print(f"\n📌 Distribuição por categoria:")
print(df_sem_dist_analise['categoria'].value_counts())

# Top 10 obras com maior distribuição
print("\n" + "=" * 120)
print("🎯 TOP 10 OBRAS COM MAIOR DISTRIBUIÇÃO (SEM DISTRIBUIDOR REGISTRADO)")
print("=" * 120)

top10 = df_sem_dist_analise.nlargest(10, 'total_sessoes')[[
    'cpb_roe', 'titulo', 'primeira_sessao', 'ultima_sessao', 
    'total_sessoes', 'salas', 'complexos', 'publico_total', 'duracao_dias'
]].copy()

# Formatar datas
top10['primeira_sessao'] = pd.to_datetime(top10['primeira_sessao']).dt.date
top10['ultima_sessao'] = pd.to_datetime(top10['ultima_sessao']).dt.date

# Renomear colunas para exibição
top10.columns = [
    'CPB/ROE', 'Título', 'Primeira Exibição', 'Última Exibição',
    'Total Sessões', 'Salas', 'Complexos', 'Público Total', 'Duração (dias)'
]

print(top10.to_string(index=False))
print("=" * 120)


ANÁLISE: OBRAS SEM DISTRIBUIDOR REGISTRADO

📊 Total de obras sem distribuidor: 479
   Brasileiras (CPB): 259
   Estrangeiras (ROE): 220

📌 Distribuição por categoria:
categoria
sessao_especial           242
distribuicao_comercial    143
indefinido                 67
mostra_festival            27
Name: count, dtype: int64

🎯 TOP 10 OBRAS COM MAIOR DISTRIBUIÇÃO (SEM DISTRIBUIDOR REGISTRADO)
       CPB/ROE                                        Título Primeira Exibição Última Exibição  Total Sessões  Salas  Complexos  Público Total  Duração (dias)
B2400228400000                          ZUZUBALANDIA O FILME        2024-09-05      2024-12-24           8233    546         85           9928             110
E1500192700000       HARRY POTTER E O PRISIONEIRO DE AZKABAN        2023-07-15      2025-04-16           6709   2463        740         750215             641
E2500131700000              SUPER WINGS EM VELOCIDADE MÁXIMA        2025-08-30      2025-09-30           5528    787        507   

In [28]:
# ============================================================
# CORRELAÇÃO: Porte/Major vs Padrões de Exibição
# ============================================================

print("\n" + "=" * 120)
print("CORRELAÇÃO: DISTRIBUIDORA vs PADRÕES DE EXIBIÇÃO")
print("=" * 120)

# Converter df_obras_enriquecido para Polars para o join
df_obras_pl = pl.from_pandas(
    df_obras_enriquecido[['cpb_roe', 'distribuidora_porte', 'distribuidora_major']]
)

# Join usando Polars
obras_com_dist = df_sessoes.join(
    df_obras_pl,
    left_on='CPB_ROE',
    right_on='cpb_roe',
    how='left'
)

# Converter para Pandas para análise agregada
obras_com_dist_pd = obras_com_dist.to_pandas()

print("\n📊 Público médio por sessão:")
print(obras_com_dist_pd.groupby('distribuidora_major', dropna=False)['PUBLICO'].mean())

print("\n📊 Número de complexos únicos por porte de distribuidora:")
complexos_por_porte = obras_com_dist_pd.groupby('distribuidora_porte', dropna=False)['NOME_COMPLEXO'].nunique()
print(complexos_por_porte)

print("\n📊 Total de sessões por porte:")
sessoes_por_porte = obras_com_dist_pd.groupby('distribuidora_porte', dropna=False).size()
print(sessoes_por_porte)

print("\n📊 Público total por major:")
publico_por_major = obras_com_dist_pd.groupby('distribuidora_major', dropna=False)['PUBLICO'].sum()
print(publico_por_major)

print("=" * 120)


CORRELAÇÃO: DISTRIBUIDORA vs PADRÕES DE EXIBIÇÃO

📊 Público médio por sessão:
distribuidora_major
False    18.240680
True     32.263372
NaN      27.060440
Name: PUBLICO, dtype: float64

📊 Número de complexos únicos por porte de distribuidora:
distribuidora_porte
1.0    634
2.0    757
3.0    762
4.0    877
5.0    917
NaN    872
Name: NOME_COMPLEXO, dtype: int64

📊 Total de sessões por porte:
distribuidora_porte
1.0       35773
2.0       60267
3.0       83545
4.0      242590
5.0    11055749
NaN      134153
dtype: int64

📊 Público total por major:
distribuidora_major
False     52465432
True     277517915
NaN        3630158
Name: PUBLICO, dtype: int64
